# Lean 4 formalization of Theorem 2.2

Supplementary material for J. Humpherys, *The De Cock–De Moor Conjecture: Proof, Normal Form, and Indefinite Angles*.

This notebook is self-contained. Cell 1 runs a quick numerical check of the identities (I1), (I2) and the similarity. Cell 2 installs Lean and Mathlib (tag `v4.19.0`). Cell 3 writes the embedded Lean source, which is byte-identical to `DeCockDeMoor/Basic.lean` in this folder, and builds it. Cell 4 reports any errors and, if the build is clean, prints the axioms used by `reduction`, `identity_I1`, `identity_I2`, and `decock_demoor`.

The formalization is fully verified when every one of these theorems lists only `propext`, `Classical.choice`, and `Quot.sound`. Run all cells; the first run takes about 5 to 15 minutes.


In [ ]:
# Cell 1: numerical check of (I1), (I2) and the similarity
import numpy as np
def stein(A, B, C):            # solve X - A X B = C
    n = A.shape[0]
    M = np.eye(n*n) - np.kron(B.T, A)
    return np.linalg.solve(M, C.flatten('F')).reshape(n, n, order='F')
def peval(c, A):               # sum c[k] A^k
    X = np.zeros_like(A); Ak = np.eye(A.shape[0])
    for ck in c: X = X + ck*Ak; Ak = Ak @ A
    return X
rng = np.random.default_rng(2026)
worst = 0.0
for n in [1, 2, 3, 4, 5]:
    for kind in ['random', 'jordan']:
        A = rng.normal(size=(n, n))*0.5
        if kind == 'jordan':
            A = 0.4*np.eye(n) + 0.5*np.diag(np.ones(n-1), 1)
        v = rng.normal(size=n); w = rng.normal(size=n); B = A + np.outer(v, w)
        P = stein(A, A.T, np.outer(v, v)); Q = stein(B.T, B, np.outer(w, w)); R = stein(A, B, np.outer(v, w))
        p = np.poly(B)[::-1]                     # det(zI-B), low -> high
        F = peval(p, A) @ np.linalg.inv(peval(p[::-1], A))   # p_B(A) q_B(A)^{-1}
        iQ = np.linalg.inv(Q); iP = np.linalg.inv(P)
        sc = max(1, np.abs(P).max(), np.abs(iQ).max())
        e1 = np.abs(R@iQ@R.T - (P - F@P@F.T)).max()/sc
        e2 = np.abs(F@(P+iQ)@F.T - P).max()/sc
        U = iP@F
        X = iP@R@iQ@R.T; Y = np.linalg.inv(np.eye(n)+P@Q)
        e3 = np.abs(np.linalg.solve(U, X@U) - Y).max()/max(1, np.abs(Y).max())
        worst = max(worst, e1, e2)
        print(f"n={n} {kind:7s}  I1 {e1:.1e}  I2 {e2:.1e}  similarity(rel) {e3:.1e}  cond(P) {np.linalg.cond(P):.1e}")
print("max relative residual of I1, I2:", worst)


In [ ]:
# Cell 2: install Lean + Mathlib (first run ~5-15 min; skipped if already present)
import os, subprocess, pathlib
MATHLIB_REV = "v4.19.0"
HOME = pathlib.Path.home()
os.environ["PATH"] = f"{HOME}/.elan/bin:" + os.environ["PATH"]
PROJ = pathlib.Path("/content/DeCockDeMoor") if pathlib.Path("/content").exists() else HOME/"DeCockDeMoor"

def run(cmd, cwd=None, check=True, quiet=False):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if not quiet: print(p.stdout[-3000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.stdout

if not (HOME/".elan/bin/elan").exists():
    run("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh "
        "| sh -s -- -y --default-toolchain none")
PROJ.mkdir(parents=True, exist_ok=True)
(PROJ/"DeCockDeMoor").mkdir(exist_ok=True)
if not (PROJ/".lake/packages/mathlib").exists():
    run(f"curl -sSfL https://raw.githubusercontent.com/leanprover-community/mathlib4/{MATHLIB_REV}/lean-toolchain -o lean-toolchain", cwd=PROJ)
    (PROJ/"lakefile.toml").write_text(f"""name = "DeCockDeMoor"
defaultTargets = ["DeCockDeMoor"]

[[require]]
name = "mathlib"
git = "https://github.com/leanprover-community/mathlib4"
rev = "{MATHLIB_REV}"

[[lean_lib]]
name = "DeCockDeMoor"
""")
    run("lake update", cwd=PROJ)
    run("lake exe cache get", cwd=PROJ)
else:
    print("Mathlib already installed in", PROJ)


In [ ]:
# Cell 3: write the Lean source (embedded) and build
import base64
LEAN_B64 = "..."
LEAN_SRC = base64.b64decode(LEAN_B64).decode()
(PROJ/"DeCockDeMoor"/"Basic.lean").write_text(LEAN_SRC)
(PROJ/"DeCockDeMoor.lean").write_text("import DeCockDeMoor.Basic\n")
(PROJ/"CheckAxioms.lean").write_text(
    "import DeCockDeMoor\n"
    "#print axioms DeCockDeMoor.reduction\n"
    "#print axioms DeCockDeMoor.identity_I1\n"
    "#print axioms DeCockDeMoor.identity_I2\n"
    "#print axioms DeCockDeMoor.decock_demoor\n")
build_log = run("lake build 2>&1", cwd=PROJ, check=False, quiet=True)


In [ ]:
# Cell 4: summary, errors grouped by declaration
import re, collections
lines = LEAN_SRC.splitlines()
decl_at = []
cur = "(header)"
for i, l in enumerate(lines, 1):
    m = re.match(r'\s*(?:noncomputable\s+)?(?:private\s+)?(lemma|theorem|def)\s+(\S+)', l)
    if m: cur = m.group(2)
    decl_at.append(cur)
blocks = re.split(r'\n(?=(?:error|warning): )', build_log)
errs = [b for b in blocks if b.startswith("error: ") and "Basic.lean" in b]
other = [b for b in blocks if b.startswith("error: ") and "Basic.lean" not in b]
by_decl = collections.OrderedDict()
for b in errs:
    m = re.search(r'Basic\.lean:(\d+):(\d+):', b)
    ln = int(m.group(1)) if m else 0
    d = decl_at[ln-1] if 0 < ln <= len(decl_at) else "?"
    by_decl.setdefault(d, []).append(b.strip()[:700])
print(f"errors in Basic.lean: {len(errs)} across {len(by_decl)} declarations")
for d, bs in by_decl.items():
    print(f"\n=== {d} ({len(bs)}) ===")
    for b in bs[:3]: print(b); print("---")
for b in other[:5]: print(b[:500])
sorry = [b for b in blocks if "sorry" in b]
print(f"\nsorry warnings: {len(sorry)}")
if not errs and not other:
    run("lake env lean CheckAxioms.lean", cwd=PROJ, check=False)
    print("Fully verified iff decock_demoor lists only [propext, Classical.choice, Quot.sound].")
else:
    print("\nPaste this cell's output back to fix the errors.")
